In [ ]:
# Block for missing dependencies
!pip install scipy
!pip install torch

In [13]:
# Import necessary libraries
import numpy as np
from scipy.linalg import expm
from scipy.optimize import minimize_scalar
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import random
from itertools import combinations_with_replacement, combinations

In [22]:
def read_matrix_from_file(filename):
    return np.load(filename)
    
def generate_spin_combinations(length):
    spin_up_indices = [i for i in range(0, length, 2)]
    spin_down_indices = [i for i in range(1, length, 2)]
    
    # Generate all unique (i, j) pairs for spin-up where i != j
    spin_up_pairs = [pair for pair in combinations(spin_up_indices, 2)]
    
    # Generate all unique (i, j) pairs for spin-down where i != j
    spin_down_pairs = [pair for pair in combinations(spin_down_indices, 2)]
    spin_up_down_pairs = list(combinations_with_replacement(spin_up_indices + spin_down_indices, 2))
    valid_spin_up_down_pairs = [pair for pair in spin_up_down_pairs if (pair[0] in spin_up_indices and pair[1] in spin_down_indices) or (pair[0] in spin_down_indices and pair[1] in spin_up_indices)]
    
    # Generate all valid (i, j, k, l) combinations for spin-up and spin-down
    valid_combinations = []

    for ij in spin_up_pairs:
        for kl in spin_up_pairs:
            valid_combinations.append((ij[0], ij[1], kl[0], kl[1]))

    for ij in spin_down_pairs:
        for kl in spin_down_pairs:
            valid_combinations.append((ij[0], ij[1], kl[0], kl[1]))

    for ij in valid_spin_up_down_pairs:
        for kl in valid_spin_up_down_pairs:
            valid_combinations.append((ij[0], ij[1], kl[0], kl[1]))

    return valid_combinations

sigma_x = np.array([[0, 1]  ,  [1, 0]] , dtype=complex)
sigma_y = np.array([[0, -1j], [1j, 0]] , dtype=complex)
sigma_z = np.array([[1, 0]  , [0, -1]] , dtype=complex)
iden    = np.array([[1, 0]  ,  [0, 1]] , dtype=complex)

# Qubit tomography is used here, no difference if theta is varied
def anni(pos, N=4):
    mat = 1
    for i in range(N):
        if i<pos or i>pos :
            mat = np.kron(mat, iden)
        if i==pos :
            mat = np.kron(mat, (sigma_x+1j*sigma_y)/2)
    return mat

def crea(pos, N=4):
    mat = 1
    for i in range(N):
        if i<pos or i>pos :
            mat = np.kron(mat, iden)
        if i==pos :
            mat = np.kron(mat, (sigma_x-1j*sigma_y)/2)
    return mat

def rdmtomo(i,j,k,l, N=4):
    tomo = anni(k, N)
    tomo = np.matmul(anni(l, N), tomo)
    tomo = np.matmul(crea(j, N), tomo)
    tomo = np.matmul(crea(i, N), tomo)
    return tomo

def generate_tomo_list(index_list, norbit):
    mat_list = []
    for item in index_list:
        mat_item = rdmtomo(item[0],item[1],item[2],item[3], norbit)
        mat_list.append(mat_item)
    return mat_list
    
def energy(wf, H_mat):
    def is_effectively_real(number, tol=1e-10):
        return np.isclose(number.imag, 0, atol=tol)

    result = np.dot(np.conj(wf.T), np.dot(H_mat, wf))

    if not is_effectively_real(result):
        raise ValueError(f"Computed energy has a non-negligible imaginary component: {result}")

    return result.real

def apply_discrete_RDM_operator(wavefunction, tomo_matrix_list, i, H_mat):
    def objective(epsilon):
        anti_commute_matrix = np.dot(tomo_matrix_list[i], H_mat) - np.dot(H_mat, tomo_matrix_list[i])
        anti_commute_value = np.dot(np.conj(wavefunction.T), np.dot(anti_commute_matrix, wavefunction))
        operator = expm(epsilon * anti_commute_value * tomo_matrix_list[i])
        new_wf = np.dot(operator, wavefunction)
        new_wf /= np.linalg.norm(new_wf)
        return energy(new_wf, H_mat)

    result = minimize_scalar(objective, method='brent')

    # Compute optimized wavefunction
    optimal_epsilon = result.x
    anti_commute_matrix = np.dot(tomo_matrix_list[i], H_mat) - np.dot(H_mat, tomo_matrix_list[i])
    anti_commute_value = np.dot(np.conj(wavefunction.T), np.dot(anti_commute_matrix, wavefunction))
    operator = expm(optimal_epsilon * anti_commute_value * tomo_matrix_list[i])
    optimal_wf = np.dot(operator, wavefunction)
    optimal_wf /= np.linalg.norm(optimal_wf)

    return optimal_wf
def anti_commute(wf, tomo_matrix_list, H_mat):
    reward = []
    for item in tomo_matrix_list:
        anti_commute_matrix = np.dot(item, H_mat) - np.dot(H_mat, item)
        reward.append(np.dot(np.conj(wf.T), np.dot(anti_commute_matrix, wf)))
    return np.array(reward)

def generate_random_wavefunction(num_orbitals=6):
    wavefunction = np.zeros(2**num_orbitals, dtype=complex)

    valid_indices = []
    for state in combinations(range(num_orbitals), 3):
        binary_state = [0] * num_orbitals
        for idx in state:
            binary_state[idx] = 1
        up_count = sum(binary_state[i] for i in range(0, num_orbitals, 2))
        down_count = sum(binary_state[i] for i in range(1, num_orbitals, 2))

        # Ensure exactly two spin-up and two spin-down electrons
        if up_count == 2 and down_count == 1:  #change this for H4
            index = int("".join(map(str, binary_state)), 2)
            valid_indices.append(index)

    coefficients = np.random.randn(len(valid_indices))
    norm = np.linalg.norm(coefficients)
    coefficients /= norm
    for i, index in enumerate(valid_indices):
        wavefunction[index] = coefficients[i]

    return wavefunction

class D3QNNetwork(nn.Module):
    def __init__(self, state_dim, num_discrete_actions, hidden_dim=256):
        super(D3QNNetwork, self).__init__()

        self.shared_layers = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

        # Separate networks for value and advantage (Dueling DQN)
        self.value_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1)
        )

        self.advantage_stream = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, num_discrete_actions)
        )

        # Initialize weights
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.orthogonal_(module.weight, gain=np.sqrt(2))
            module.bias.data.fill_(0.0)

    def forward(self, state):
        shared_features = self.shared_layers(state)

        # Dueling DQN computation
        value = self.value_stream(shared_features)
        advantage = self.advantage_stream(shared_features)
        q_values = value + (advantage - advantage.mean(dim=1, keepdim=True))

        return q_values

class PrioritizedReplayBuffer:
    def __init__(self, capacity, alpha=0.6):
        self.capacity = capacity
        self.alpha = alpha
        self.buffer = []
        self.priorities = np.zeros(capacity)
        self.position = 0

    def push(self, state, discrete_action, reward, next_state, done):
        max_priority = self.priorities.max() if self.buffer else 1.0

        if len(self.buffer) < self.capacity:
            self.buffer.append((state, discrete_action, reward, next_state, done))
        else:
            self.buffer[self.position] = (state, discrete_action, reward, next_state, done)

        self.priorities[self.position] = max_priority
        self.position = (self.position + 1) % self.capacity

    def sample(self, batch_size, beta):
        if len(self.buffer) < batch_size:
            return None, None, None

        # Calculate sampling probabilities
        priorities = self.priorities[:len(self.buffer)]
        probabilities = priorities ** self.alpha
        probabilities /= probabilities.sum()

        # Sample indices and calculate importance sampling weights
        indices = np.random.choice(len(self.buffer), batch_size, p=probabilities)
        weights = (len(self.buffer) * probabilities[indices]) ** (-beta)
        weights /= weights.max()

        batch = [self.buffer[idx] for idx in indices]
        states, discrete_actions, rewards, next_states, dones = zip(*batch)

        return (np.array(states),
                np.array(discrete_actions),
                np.array(rewards),
                np.array(next_states),
                np.array(dones)), weights, indices

    def update_priorities(self, indices, td_errors):
        for idx, error in zip(indices, td_errors):
            self.priorities[idx] = error[0] + 1e-6  # Small constant to prevent zero priority

    def __len__(self):
        return len(self.buffer)

class D3QN:
    def __init__(self, state_dim, num_discrete_actions):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Using device: {self.device}")

        # Initialize networks with improved architecture
        self.policy_net = D3QNNetwork(state_dim, num_discrete_actions).to(self.device)
        self.target_net = D3QNNetwork(state_dim, num_discrete_actions).to(self.device)
        self.target_net.load_state_dict(self.policy_net.state_dict())

        # Use AdamW optimizer with weight decay
        self.optimizer = optim.AdamW(self.policy_net.parameters(), lr=3e-4, weight_decay=1e-4)

        # Learning rate scheduler
        self.scheduler = optim.lr_scheduler.CosineAnnealingLR(self.optimizer, T_max=1000, eta_min=1e-5)

        # Prioritized Experience Replay (change if necessary)
        self.memory = PrioritizedReplayBuffer(capacity=5000000, alpha=0.6)

        self.num_discrete_actions = num_discrete_actions

        # Hyperparameters
        self.batch_size = 256
        self.gamma = 0.99
        self.epsilon = 1.0
        self.epsilon_min = 0.01  # Lower minimum epsilon
        self.epsilon_decay = 0.995  # For demonstration 0.995 is used, which decay in ~900 episode
        self.target_update = 10  #change this as well
        self.beta = 0.4
        self.beta_increment = 0.001

        # Training metrics
        self.metrics = {
            'episode_rewards': [],
            'episode_energies': [],
            'episode_residuals': [],
            'episode_steps': [],
            'losses': []
        }

    def select_action(self, state, training=True):
        state = torch.FloatTensor(state).unsqueeze(0).to(self.device)

        if training and random.random() < self.epsilon:
            discrete_action = random.randrange(self.num_discrete_actions)
        else:
            with torch.no_grad():
                q_values = self.policy_net(state)
                discrete_action = q_values.max(1)[1].item()

        return discrete_action

    def _update_network(self):
        if len(self.memory) < self.batch_size:
            return

        # Sample with priorities and importance sampling
        batch, weights, indices = self.memory.sample(self.batch_size, self.beta)
        states, discrete_actions, rewards, next_states, dones = batch

        # Convert to tensors and move to device
        states = torch.FloatTensor(states).to(self.device)
        discrete_actions = torch.LongTensor(discrete_actions).to(self.device)
        rewards = torch.FloatTensor(rewards).to(self.device)
        next_states = torch.FloatTensor(next_states).to(self.device)
        dones = torch.FloatTensor(dones).to(self.device)
        weights = torch.FloatTensor(weights).to(self.device)

        # Double DQN: Use policy net to select actions, target net to evaluate them
        with torch.no_grad():
            next_q_values = self.policy_net(next_states)
            next_actions = next_q_values.max(1)[1]

            next_q_values = self.target_net(next_states)
            next_q_values = next_q_values.gather(1, next_actions.unsqueeze(1))

        # Compute target Q values
        target_q_values = rewards + (1 - dones) * self.gamma * next_q_values

        # Get current Q values
        current_q_values = self.policy_net(states)
        current_q_values = current_q_values.gather(1, discrete_actions.unsqueeze(1))

        # Compute losses with importance sampling weights
        q_loss = (weights * F.smooth_l1_loss(current_q_values, target_q_values, reduction='none')).mean()

        # Compute priorities for replay buffer
        with torch.no_grad():
            td_errors = torch.abs(target_q_values - current_q_values).cpu().numpy()
            self.memory.update_priorities(indices, td_errors)
        total_loss = q_loss

        # Optimize
        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_net.parameters(), max_norm=10.0)
        self.optimizer.step()
        self.scheduler.step()

        # Update beta for importance sampling
        self.beta = min(1.0, self.beta + self.beta_increment)
        self.metrics['losses'].append(total_loss.item())

    def train(self, env, num_episodes):
        for episode in range(num_episodes):
            state = env.reset()
            total_reward = 0
            done = False
            step = 0
            #use step = 5 as terminatation, alternatively one can use energy/residual
            while not done and step < 5:
                step += 1
                discrete_action = self.select_action(state)
                next_state, E, residuals, done = env.step(discrete_action)
                reward = -E - 0.2*np.sum(np.abs(residuals))

                # Store transition in memory
                self.memory.push(state, discrete_action, [reward], next_state, [done])

                # Train if enough samples
                if len(self.memory) > self.batch_size:
                    self._update_network()

                state = next_state
                total_reward += reward

            # Store metrics: average rewards per episode per step is used
            self.metrics['episode_rewards'].append(total_reward/step)
            self.metrics['episode_energies'].append(E)
            self.metrics['episode_residuals'].append(np.mean(np.abs(state)))
            self.metrics['episode_steps'].append(step)

            # Decay exploration rate
            self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)

            # Update target network
            if episode % self.target_update == 0:
                self.target_net.load_state_dict(self.policy_net.state_dict())

            # Print progress per n episode
            if episode % 100 == 0:
                print(f"Episode {episode}: Final Energy: {E:.4f}, Steps: {step}")

class QuantumStateEnv:
    def __init__(self, initial_wf, H_mat, tomo_matrix_list):
        self.initial_wf = initial_wf
        self.H_mat = H_mat
        self.tomo_matrix_list = tomo_matrix_list
        self.current_wf = initial_wf.copy()

    def reset(self):
        self.current_wf = self.initial_wf.copy()
        return self.get_state_vector()

    def step(self, discrete_action):
        # Apply the quantum operation
        self.current_wf = apply_discrete_RDM_operator(
            self.current_wf,
            self.tomo_matrix_list,
            discrete_action,
            self.H_mat
        )

        # Get new state
        new_state = self.get_state_vector()

        # Calculate reward
        E = energy(self.current_wf, self.H_mat)
        residuals = new_state

        # Done condition. Not needed for small number of actions
        done = np.all(np.abs(residuals) < 1e-4)

        return new_state, E, residuals, done

    def get_state_vector(self):
        residuals = anti_commute(self.current_wf, self.tomo_matrix_list, self.H_mat).real
        return residuals

In [21]:
if __name__ == '__main__':
    hamiltonian_matrix = read_matrix_from_file("H3_example_hamiltonian.npy")
    norbit = 6
    index_list = generate_spin_combinations(norbit)
    tomo_matrix_list = generate_tomo_list(index_list, norbit)

    initial_wavefunction = generate_random_wavefunction()
    print ("Starting energy is: ", energy(initial_wavefunction, hamiltonian_matrix))

    # Environment setup
    env = QuantumStateEnv(initial_wavefunction, hamiltonian_matrix, tomo_matrix_list)
    num_discrete_actions = len(tomo_matrix_list)

    # D3QN Agent
    state_dim = env.get_state_vector().shape[0]
    print ("State_dim is ", state_dim)
    agent = D3QN(state_dim, num_discrete_actions)

    num_episodes = 2000
    agent.train(env, num_episodes)

Starting energy is:  -0.9648414985570206
State_dim is  99
Using device: cpu
Episode 0: Final Energy: -1.1521, Steps: 5
Episode 100: Final Energy: -1.1046, Steps: 5
Episode 200: Final Energy: -1.3762, Steps: 5
Episode 300: Final Energy: -1.3906, Steps: 5
Episode 400: Final Energy: -1.3921, Steps: 5
Episode 500: Final Energy: -1.3925, Steps: 5
Episode 600: Final Energy: -1.3907, Steps: 5
Episode 700: Final Energy: -1.3917, Steps: 5
Episode 800: Final Energy: -1.3904, Steps: 5
Episode 900: Final Energy: -1.2318, Steps: 5
Episode 1000: Final Energy: -1.3955, Steps: 5
Episode 1100: Final Energy: -1.3904, Steps: 5
Episode 1200: Final Energy: -1.3955, Steps: 5
Episode 1300: Final Energy: -1.3917, Steps: 5
Episode 1400: Final Energy: -1.3948, Steps: 5
Episode 1500: Final Energy: -1.3964, Steps: 5
Episode 1600: Final Energy: -0.9874, Steps: 5
Episode 1700: Final Energy: -1.3966, Steps: 5
Episode 1800: Final Energy: -1.3967, Steps: 5
Episode 1900: Final Energy: -1.3967, Steps: 5
